# Notebook 00 — Setup and Installation

**Run this once before anything else.**

- Installs all packages
- Downloads MedSAM checkpoint
- Sets project paths
- Verifies dataset and GPU

In [1]:
import subprocess, sys

pkgs = [
    'timm==0.9.16',
    'SimpleITK',
    'scikit-learn',
    'matplotlib',
    'pandas',
    'openpyxl',
    'scipy',
    'tqdm',
    'pyyaml',
    'git+https://github.com/facebookresearch/segment-anything.git',
]
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True
    )
    status = 'OK' if r.returncode == 0 else f'FAIL: {r.stderr.decode()[:80]}'
    print(f'{pkg[:40]:<40} {status}')

print('\nAll packages done.')

timm==0.9.16                             OK
SimpleITK                                OK
scikit-learn                             OK
matplotlib                               OK
pandas                                   OK
openpyxl                                 OK
scipy                                    OK
tqdm                                     OK
pyyaml                                   OK
git+https://github.com/facebookresearch/ OK

All packages done.


In [2]:
import torch

assert torch.cuda.is_available(), 'CUDA not found. Check GPU drivers.'
props = torch.cuda.get_device_properties(0)
vram  = props.total_memory / 1e9

print(f'GPU   : {props.name}')
print(f'VRAM  : {vram:.1f} GB')
print(f'CUDA  : {torch.version.cuda}')
print(f'Torch : {torch.__version__}')

if vram < 10:
    print('WARN: < 10 GB. Set batch_size=2 in medsam.yaml')
elif vram < 12:
    print('WARN: 10-12 GB. batch_size=4 should be fine with AMP.')
else:
    print('OK: VRAM sufficient for all models.')

GPU   : NVIDIA GeForce RTX 4070
VRAM  : 12.9 GB
CUDA  : 12.1
Torch : 2.5.1+cu121
OK: VRAM sufficient for all models.


In [4]:
import json
from pathlib import Path

# ─── SET THESE PATHS ───────────────────────────────

PROJECT_ROOT = Path('.').resolve().parent

FED7_ROOT = Path(
    r'C:\Users\Mayank\Documents\fedbca_project'
)

DATA_ROOT = Path(
    r'C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\FedBCa'
)

PREPROCESSED = PROJECT_ROOT / 'preprocessed'

MEDSAM_CKPT = (
    PROJECT_ROOT /
    'checkpoints' /
    'medsam_vit_b.pth'
)

EXPERIMENTS = PROJECT_ROOT / 'experiments'

# ───────────────────────────────────────────────────

for d in [
    PREPROCESSED,
    EXPERIMENTS,
    MEDSAM_CKPT.parent
]:
    d.mkdir(parents=True, exist_ok=True)

paths = {
    'project_root': str(PROJECT_ROOT),
    'fed7_root': str(FED7_ROOT),
    'data_root': str(DATA_ROOT),
    'preprocessed': str(PREPROCESSED),
    'medsam_ckpt': str(MEDSAM_CKPT),
    'experiments': str(EXPERIMENTS),
}

(PROJECT_ROOT / 'paths.json').write_text(
    json.dumps(paths, indent=2)
)

print('paths.json saved.\n')

print(json.dumps(paths, indent=2))

paths.json saved.

{
  "project_root": "C:\\Users\\Mayank\\Desktop\\fedbca_medsam_complete\\fedbca_medsam",
  "fed7_root": "C:\\Users\\Mayank\\Documents\\fedbca_project",
  "data_root": "C:\\Users\\Mayank\\Desktop\\fedbca_medsam_complete\\fedbca_medsam\\FedBCa",
  "preprocessed": "C:\\Users\\Mayank\\Desktop\\fedbca_medsam_complete\\fedbca_medsam\\preprocessed",
  "medsam_ckpt": "C:\\Users\\Mayank\\Desktop\\fedbca_medsam_complete\\fedbca_medsam\\checkpoints\\medsam_vit_b.pth",
  "experiments": "C:\\Users\\Mayank\\Desktop\\fedbca_medsam_complete\\fedbca_medsam\\experiments"
}


In [8]:
import urllib.request
from pathlib import Path
import torch

# =========================================================
# CHECKPOINT PATH
# =========================================================

MEDSAM_CKPT = Path(paths['medsam_ckpt'])

# =========================================================
# DOWNLOAD URL
# =========================================================

MEDSAM_URL = (
    'https://huggingface.co/spaces/junma/medsam/'
    'resolve/main/work_dir/MedSAM/medsam_vit_b.pth'
)

# =========================================================
# DOWNLOAD CHECKPOINT
# =========================================================

if MEDSAM_CKPT.exists():

    size_mb = MEDSAM_CKPT.stat().st_size / 1e6

    print(
        f'Checkpoint already exists:\n'
        f'{MEDSAM_CKPT}\n'
        f'({size_mb:.0f} MB)'
    )

else:

    print('Downloading MedSAM checkpoint...\n')

    print(f'Target path:\n{MEDSAM_CKPT}\n')

    print('If download fails, download manually from:')
    print('https://github.com/bowang-lab/MedSAM\n')

    downloaded = [0]

    # -----------------------------------------------------
    # PROGRESS HOOK
    # -----------------------------------------------------

    def hook(count, block_size, total_size):

        downloaded[0] = count * block_size

        if total_size > 0:

            pct = min(
                100,
                100 * downloaded[0] / total_size
            )

            print(
                f'\r'
                f'{pct:.1f}% '
                f'| '
                f'{downloaded[0]/1e6:.0f}/'
                f'{total_size/1e6:.0f} MB',
                end=''
            )

    # -----------------------------------------------------
    # DOWNLOAD
    # -----------------------------------------------------

    try:

        urllib.request.urlretrieve(
            MEDSAM_URL,
            str(MEDSAM_CKPT),
            hook
        )

        print('\n\nDownload complete.')

    except Exception as e:

        print(f'\n\nDownload failed:\n{e}')

        print('\nDownload manually and place file at:')

        print(MEDSAM_CKPT)

# =========================================================
# VERIFY CHECKPOINT
# =========================================================

if MEDSAM_CKPT.exists():

    try:

        ckpt = torch.load(
            str(MEDSAM_CKPT),
            map_location='cpu'
        )

        print('\nCheckpoint loaded successfully.')

        print(
            f'First 5 keys:\n'
            f'{list(ckpt.keys())[:5]}'
        )

        print('\nMedSAM checkpoint verification OK.')

    except Exception as e:

        print('\nCheckpoint exists but failed to load.')

        print(f'Error:\n{e}')

else:

    print('\nCheckpoint not found.')

    print('Download manually before training.')

Checkpoint already exists:
C:\Users\Mayank\Desktop\fedbca_medsam_complete\fedbca_medsam\checkpoints\medsam_vit_b.pth
(375 MB)

Checkpoint loaded successfully.
First 5 keys:
['image_encoder.neck.0.weight', 'image_encoder.neck.1.weight', 'image_encoder.neck.1.bias', 'image_encoder.neck.2.weight', 'image_encoder.neck.3.weight']

MedSAM checkpoint verification OK.


C:\Users\Mayank\AppData\Local\Temp\ipykernel_1014696\3243915115.py:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(


In [10]:
from pathlib import Path

DATA_ROOT = Path(paths['data_root'])

print('Checking FedBCa dataset structure...')
all_ok = True

for c in range(1, 5):
    center = DATA_ROOT / f'Center{c}'

    # Try multiple naming conventions
    img_dir = next(
        (center / d for d in ['Image', 'T2WI', 'image']
         if (center / d).exists()), None
    )
    ann_dir = next(
        (center / d for d in ['Annotation', 'annotation', 'Mask']
         if (center / d).exists()), None
    )
    lbl = next(
        (center / f for f in [
            f'Center_{c}_label.xlsx',
            f'center{c}_label.xlsx',
            'label.xlsx'
        ] if (center / f).exists()), None
    )

    n_img = len(list(img_dir.glob('*.nii.gz'))) if img_dir else 0
    n_ann = len(list(ann_dir.glob('*.nii.gz'))) if ann_dir else 0
    ok = n_img > 0 and lbl is not None
    if not ok:
        all_ok = False

    print(f'  Center {c}: {n_img:3d} images | {n_ann:3d} masks | '
          f'label={lbl.name if lbl else "MISSING"} | {"OK" if ok else "MISSING"}')

if all_ok:
    print('\nDataset OK. Proceed to Notebook 01.')
else:
    print('\nERROR: Missing files. Download from:')
    print('  https://zenodo.org/records/10409145')

Checking FedBCa dataset structure...
  Center 1: 119 images | 160 masks | label=Center_1_label.xlsx | OK
  Center 2:  47 images |  48 masks | label=Center_2_label.xlsx | OK
  Center 3:  24 images |  32 masks | label=Center_3_label.xlsx | OK
  Center 4:  31 images |  35 masks | label=Center_4_label.xlsx | OK

Dataset OK. Proceed to Notebook 01.
